In [ ]:
# 待办
京东|https://zhaopin.jd.com/web/job/job_info_list/3
深蓝互动|https://app.mokahr.com/apply/blueinteractive/38433#/jobs?zhineng=68723

# 测试

## 蛮啾网络

### 方法2

In [1]:
import json
import time
import datetime
from selenium import webdriver
from selenium.webdriver.edge.options import Options

def fetch_manjiu_top3_jobs():
    # ===================== 核心配置 =====================
    target_page_url = "https://ooia5293gn.jobs.feishu.cn/index/position/list?keywords=&category=&location=CT_125&project=&type=&job_hot_flag=&current=1&limit=10&functionCategory=7510139416967448895&tag="
    target_api_key = "/api/v1/search/job/posts"
    detail_url_prefix = "https://ooia5293gn.jobs.feishu.cn/index/position/"

    # ===================== 浏览器配置 =====================
    edge_options = Options()
    edge_options.add_argument("--headless=new")
    edge_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    edge_options.add_argument("--disable-blink-features=AutomationControlled")

    # 启动浏览器
    driver = webdriver.Edge(options=edge_options)

    # ===================== JS注入拦截器 =====================
    def inject_api_interceptor(driver_instance):
        driver_instance.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
            "source": """
                window.CAPTURE_API_DATA = null;
                // 拦截 Fetch
                const rawFetch = window.fetch;
                window.fetch = async (...args) => {
                    const resp = await rawFetch(...args);
                    if(args[0].includes('""" + target_api_key + """')){
                        window.CAPTURE_API_DATA = await resp.clone().json();
                    }
                    return resp;
                };
                // 拦截 XHR (备用，防止页面用XHR)
                const rawXHR = window.XMLHttpRequest;
                window.XMLHttpRequest = function(){
                    const xhr = new rawXHR();
                    xhr.addEventListener('load', ()=>{
                        if(xhr.responseURL.includes('""" + target_api_key + """')){
                            try{ window.CAPTURE_API_DATA = JSON.parse(xhr.responseText); }catch{}
                        }
                    });
                    return xhr;
                };
            """
        })

    try:
        # 1. 先注入脚本，再访问页面
        inject_api_interceptor(driver)
        print("[蛮啾] 正在加载页面...")
        driver.get(target_page_url)
        
        # 2. 等待数据加载 (JS劫持需要时间)
        time.sleep(3)

        # 3. 从 window 对象中取出拦截到的数据
        api_data = driver.execute_script("return window.CAPTURE_API_DATA")
        
        if not api_data:
            print("未能捕获到API数据，请检查网络或页面结构是否变化")
            return

        # 4. 解析数据 (逻辑与原代码保持一致)
        job_list = api_data.get("data", {}).get("job_post_list", [])

        if not job_list:
            print("接口返回中未找到岗位列表")
            return

        # ===================== 格式化输出 =====================
        print("="*60)
        print("蛮啾网络 上海 运营商务类 前3个岗位信息")
        print("="*60)

        for idx, job in enumerate(job_list[:3], 1):
            job_id = job.get("id", "")
            job_title = job.get("title", "无岗位名称")
            job_location = job.get("city_list", [{}])[0].get("name", "无地点")
            job_address = job.get("job_post_info", {}).get("address_list", [{}])[0].get("name", "无详细地址")
            
            recruit_type = job.get("recruit_type", {})
            job_recruit_type = recruit_type.get("parent", {}).get("name", "无招聘类型")
            job_job_type = recruit_type.get("name", "无用工类型")
            job_function = job.get("job_function", {}).get("name", "无职能分类")
            job_description = job.get("description", "无岗位职责").strip()
            job_requirement = job.get("requirement", "无任职要求").strip()

            publish_timestamp = job.get("publish_time")
            if publish_timestamp:
                publish_time = datetime.datetime.fromtimestamp(publish_timestamp / 1000).strftime("%Y-%m-%d %H:%M:%S")
            else:
                publish_time = "无发布时间"

            job_detail_url = f"{detail_url_prefix}{job_id}/detail" if job_id else "无详情链接"

            print(f"\n【岗位{idx}：{job_title}】")
            print(f"🔢 岗位ID：{job_id}")
            print(f"📍 工作地点：{job_location} | 🏢 详细地址：{job_address}")
            print(f"📌 招聘类型：{job_recruit_type} | 📎 用工类型：{job_job_type}")
            print(f"📂 职能分类：{job_function} | 🕒 发布时间：{publish_time}")
            print(f"🔗 岗位详情页：{job_detail_url}")
            print("📖 岗位职责：")
            print(job_description)
            print("📋 任职要求：")
            print(job_requirement)
            print("-"*60)

    except Exception as e:
        print(f"程序运行出错：{str(e)}")
    finally:
        driver.quit()

if __name__ == "__main__":
    fetch_manjiu_top3_jobs()

The msedgedriver version (145.0.3800.82) detected in PATH at .\msedgedriver.exe might not be compatible with the detected MicrosoftEdge version (146.0.3856.97); currently, msedgedriver 146.0.3856.97 is recommended for MicrosoftEdge 146.*, so it is advised to delete the driver in PATH and retry


[蛮啾] 正在加载页面...
蛮啾网络 上海 运营商务类 前3个岗位信息

【岗位1：IP衍生品授权经理】
🔢 岗位ID：7618422966704326918
📍 工作地点：上海 | 🏢 详细地址：宜山路700号B2栋23层
📌 招聘类型：社招 | 📎 用工类型：全职
📂 职能分类：运营商务类 | 🕒 发布时间：2026-03-18 10:56:45
🔗 岗位详情页：https://ooia5293gn.jobs.feishu.cn/index/position/7618422966704326918/detail
📖 岗位职责：
1、负责游戏IP衍生品的授权企划与生产落地全流程，并协助完成对外授权衍生品的案件跟进；
2、熟悉IP衍生品用户群体，能根据IP特点和用户需求，提出IP衍生品的创意企划方案；
3、熟悉IP衍生品制作全流程，能管理并协调内外部美术及供应商等，把控产品整体开发工期进度，并保障落地品质细节；
4、熟悉IP衍生品授权业务全流程，能协调外部合作方及内部各部门，把控产品整体开发，并保障案件完成度；
5、整理并分析产品上线后的用户反馈和销售数据，优化整体用户体验。
📋 任职要求：
1、本科及以上学历，3年以上文创产品或IP衍生品行业工作经验；
2、对游戏行业有热情，了解游戏衍生品周边与IP授权行业的市场动态；
3、熟悉衍生品生产流程和材料工艺，熟悉IP授权业务流程；
4、具备良好的沟通能力及团队协作能力，有责任心；
5、对二次元游戏有热情，有碧蓝航线游戏经验者优先。
------------------------------------------------------------

【岗位2：版本运营（活动向）】
🔢 岗位ID：7611731371002038591
📍 工作地点：上海 | 🏢 详细地址：宜山路700号B2栋23层
📌 招聘类型：社招 | 📎 用工类型：全职
📂 职能分类：运营商务类 | 🕒 发布时间：2026-02-28 11:04:17
🔗 岗位详情页：https://ooia5293gn.jobs.feishu.cn/index/position/7611731371002038591/detail
📖 岗位职责：
1.深度参与游戏内运营活动的全流程管线，制定活动版本规划并协同研发团队完成需求落地、资源排期与上线验收。
2.

### 方法1

In [7]:
import json
import time
import datetime
from selenium import webdriver
from selenium.webdriver.edge.options import Options

def fetch_manjiu_top3_jobs():
    # 浏览器配置
    edge_options = Options()
    # 无头模式（后台运行，不弹出浏览器，如需观察界面可注释此行）
    edge_options.add_argument("--headless=new")
    # 开启性能日志，用于捕获网络请求
    edge_options.set_capability('ms:loggingPrefs', {'performance': 'ALL'})
    # 屏蔽自动化特征，避免页面反爬拦截
    edge_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    edge_options.add_argument("--disable-blink-features=AutomationControlled")

    # 启动Edge浏览器
    driver = webdriver.Edge(options=edge_options)
    # 目标页面地址
    target_page_url = "https://ooia5293gn.jobs.feishu.cn/index/position/list?keywords=&category=&location=CT_125&project=&type=&job_hot_flag=&current=1&limit=10&functionCategory=7510139416967448895&tag="
    # 目标API接口标识
    target_api_key = "/api/v1/search/job/posts"
    # 岗位详情页URL前缀
    detail_url_prefix = "https://ooia5293gn.jobs.feishu.cn/index/position/"

    try:
        # 开启Network监听
        driver.execute_cdp_cmd("Network.enable", {})
        # 访问目标页面，触发浏览器自动发起带合法签名的API请求
        driver.get(target_page_url)
        # 等待页面与接口请求完成
        time.sleep(3)

        # 提取性能日志，筛选目标接口的有效请求
        logs = driver.get_log("performance")
        valid_request_id = None

        for entry in logs:
            try:
                msg = json.loads(entry["message"])["message"]
                # 匹配目标接口，且只保留HTTP 200的有效响应
                if msg["method"] == "Network.responseReceived":
                    response = msg["params"]["response"]
                    if target_api_key in response["url"] and response["status"] == 200:
                        valid_request_id = msg["params"]["requestId"]
                        break
            except Exception:
                continue

        if not valid_request_id:
            print("未能捕获到有效接口请求，请检查网络或页面访问情况")
            return

        # 获取接口响应体并解析JSON
        response_body = driver.execute_cdp_cmd("Network.getResponseBody", {"requestId": valid_request_id})
        api_data = json.loads(response_body["body"])
        # 提取岗位列表（已通过调试确认接口结构）
        job_list = api_data.get("data", {}).get("job_post_list", [])

        if not job_list:
            print("接口返回中未找到岗位列表")
            return

        # 格式化打印前3个岗位信息
        print("="*60)
        print("蛮啾网络 上海 运营商务类 前3个岗位信息")
        print("="*60)

        for idx, job in enumerate(job_list[:3], 1):
            # 提取核心字段
            job_id = job.get("id", "")
            job_title = job.get("title", "无岗位名称")
            job_location = job.get("city_list", [{}])[0].get("name", "无地点")
            job_address = job.get("job_post_info", {}).get("address_list", [{}])[0].get("name", "无详细地址")
            # 招聘类型：社招/全职
            recruit_type = job.get("recruit_type", {})
            job_recruit_type = recruit_type.get("parent", {}).get("name", "无招聘类型")
            job_job_type = recruit_type.get("name", "无用工类型")
            job_function = job.get("job_function", {}).get("name", "无职能分类")
            job_description = job.get("description", "无岗位职责").strip()
            job_requirement = job.get("requirement", "无任职要求").strip()

            # 处理发布/更新时间（13位毫秒级时间戳转可读格式）
            publish_timestamp = job.get("publish_time")
            if publish_timestamp:
                publish_time = datetime.datetime.fromtimestamp(publish_timestamp / 1000).strftime("%Y-%m-%d %H:%M:%S")
            else:
                publish_time = "无发布时间"

            # 新增：拼接岗位详情页URL
            if job_id:
                job_detail_url = f"{detail_url_prefix}{job_id}/detail"
            else:
                job_detail_url = "无详情链接"

            # 格式化打印输出
            print(f"\n【岗位{idx}：{job_title}】")
            print(f"🔢 岗位ID：{job_id}")
            print(f"📍 工作地点：{job_location} | 🏢 详细地址：{job_address}")
            print(f"📌 招聘类型：{job_recruit_type} | 📎 用工类型：{job_job_type}")
            print(f"📂 职能分类：{job_function} | 🕒 发布时间：{publish_time}")
            print(f"🔗 岗位详情页：{job_detail_url}")
            print("📖 岗位职责：")
            print(job_description)
            print("📋 任职要求：")
            print(job_requirement)
            print("-"*60)

    except Exception as e:
        print(f"程序运行出错：{str(e)}")
    finally:
        # 确保浏览器正常关闭
        driver.quit()

if __name__ == "__main__":
    fetch_manjiu_top3_jobs()

The msedgedriver version (145.0.3800.82) detected in PATH at .\msedgedriver.exe might not be compatible with the detected MicrosoftEdge version (146.0.3856.97); currently, msedgedriver 146.0.3856.97 is recommended for MicrosoftEdge 146.*, so it is advised to delete the driver in PATH and retry


蛮啾网络 上海 运营商务类 前3个岗位信息

【岗位1：IP衍生品授权经理】
🔢 岗位ID：7618422966704326918
📍 工作地点：上海 | 🏢 详细地址：宜山路700号B2栋23层
📌 招聘类型：社招 | 📎 用工类型：全职
📂 职能分类：运营商务类 | 🕒 发布时间：2026-03-18 10:56:45
🔗 岗位详情页：https://ooia5293gn.jobs.feishu.cn/index/position/7618422966704326918/detail
📖 岗位职责：
1、负责游戏IP衍生品的授权企划与生产落地全流程，并协助完成对外授权衍生品的案件跟进；
2、熟悉IP衍生品用户群体，能根据IP特点和用户需求，提出IP衍生品的创意企划方案；
3、熟悉IP衍生品制作全流程，能管理并协调内外部美术及供应商等，把控产品整体开发工期进度，并保障落地品质细节；
4、熟悉IP衍生品授权业务全流程，能协调外部合作方及内部各部门，把控产品整体开发，并保障案件完成度；
5、整理并分析产品上线后的用户反馈和销售数据，优化整体用户体验。
📋 任职要求：
1、本科及以上学历，3年以上文创产品或IP衍生品行业工作经验；
2、对游戏行业有热情，了解游戏衍生品周边与IP授权行业的市场动态；
3、熟悉衍生品生产流程和材料工艺，熟悉IP授权业务流程；
4、具备良好的沟通能力及团队协作能力，有责任心；
5、对二次元游戏有热情，有碧蓝航线游戏经验者优先。
------------------------------------------------------------

【岗位2：版本运营（活动向）】
🔢 岗位ID：7611731371002038591
📍 工作地点：上海 | 🏢 详细地址：宜山路700号B2栋23层
📌 招聘类型：社招 | 📎 用工类型：全职
📂 职能分类：运营商务类 | 🕒 发布时间：2026-02-28 11:04:17
🔗 岗位详情页：https://ooia5293gn.jobs.feishu.cn/index/position/7611731371002038591/detail
📖 岗位职责：
1.深度参与游戏内运营活动的全流程管线，制定活动版本规划并协同研发团队完成需求落地、资源排期与上线验收。
2.深度挖掘用户需求与反馈，结合游

## 携程

In [ ]:
import requests
import re

# 清洗 HTML 标签
def strip_html(html):
    if not html:
        return ""
    # 去掉所有 HTML 标签
    text = re.sub(r'<[^>]+>', '', html)
    # 多余空格换行精简
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 提取「任职资格」部分
def get_requirement_text(requirements_html):
    text = strip_html(requirements_html)
    # 按关键词分割，取后半部分
    if "任职资格" in text:
        return text.split("任职资格", 1)[1].strip()
    elif "Qualifications" in text:
        return text.split("Qualifications", 1)[1].strip()
    elif "Requirements" in text:
        return text.split("Requirements", 1)[1].strip()
    return text

# 接口信息
API_URL = "https://job.ctrip.com/api/hrrecruit/getJobAd"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": "https://job.ctrip.com/",
    "Content-Type": "application/json",
    "Origin": "https://job.ctrip.com"
}

PARAMS = {
    "condition": {
        "jobCategoryType": "JFG_51,Categroy_4",
        "cityCode": "CO0009,CO0010",
        "kind": [1]
    },
    "pageIndex": 1,
    "pageSize": 10,
    "channelId": "1",
    "sortField": "publishTime",
    "sortType": "DESC"
}

def get_ctrip_jobs():
    try:
        response = requests.post(API_URL, headers=HEADERS, json=PARAMS, timeout=15)
        data = response.json()

        if data.get("retCode") != "201":
            print(f"接口调用失败：{data.get('retMessage')}")
            return

        job_list = data["retValue"]["recruitJobAdList"]
        total = data["retValue"]["total"]
        print(f"✅ 成功获取 {total} 个岗位，展示第1页：\n")

        for i, job in enumerate(job_list, 1):
            job_title = job.get("jobTitle", "")
            city_name = job.get("cityName", "")
            publish_date = job.get("publishDate", "")
            bu_name = job.get("buName", "")
            from_id = job.get("fromId", "")
            requirements = job.get("requirements", "")

            # 拼接详情页 URL
            detail_url = f"https://job.ctrip.com/#/experienced/job-detail/{from_id}"
            # 提取任职资格
            qualification = get_requirement_text(requirements)

            print(f"===== 第 {i} 个岗位 =====")
            print(f"岗位名称：{job_title}")
            print(f"城市：{city_name}")
            print(f"发布时间：{publish_date}")
            print(f"部门：{bu_name}")
            print(f"详情链接：{detail_url}")
            print(f"任职资格：{qualification[:300]}..." if len(qualification) > 300 else f"任职资格：{qualification}")
            print("-" * 80)

    except Exception as e:
        print(f"❌ 异常：{str(e)}")

if __name__ == "__main__":
    get_ctrip_jobs()

接口调用失败：Could not read document: Cannot deserialize instance of `java.util.ArrayList<java.lang.Object>` out of VALUE_STRING token
 at [Source: (PushbackInputStream); line: 1, column: 36] (through reference chain: com.ctrip.it.hrplus.recruit.service.soa.GetJobAdRequestType["condition"]->com.ctrip.it.hrplus.recruit.service.soa.GetJobAdCondition["jobFamilyGroupCode"]); nested exception is com.fasterxml.jackson.databind.exc.MismatchedInputException: Cannot deserialize instance of `java.util.ArrayList<java.lang.Object>` out of VALUE_STRING token
 at [Source: (PushbackInputStream); line: 1, column: 36] (through reference chain: com.ctrip.it.hrplus.recruit.service.soa.GetJobAdRequestType["condition"]->com.ctrip.it.hrplus.recruit.service.soa.GetJobAdCondition["jobFamilyGroupCode"])


In [5]:
import requests

# 携程招聘官方接口
API_URL = "https://job.ctrip.com/api/hrrecruit/getJobAd"

# 请求头
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": "https://job.ctrip.com/",
    "Content-Type": "application/json",
    "Origin": "https://job.ctrip.com"
}

# ✅ 终极修复：kind 必须是数组 [1]，不能是数字 1
PARAMS = {
    "condition": {
        "jobCategoryType": "JFG_51,Categroy_4",
        "cityCode": "CO0009,CO0010",
        "kind": [1]  # 这里改成数组！！！
    },
    "pageIndex": 1,
    "pageSize": 10,
    "channelId": "1",
    "sortField": "publishTime",
    "sortType": "DESC"
}

def get_ctrip_jobs():
    try:
        response = requests.post(API_URL, headers=HEADERS, json=PARAMS, timeout=15)
        data = response.json()

        if data.get("retCode") != "201":
            print(f"接口调用失败：{data.get('retMessage')}")
            return

        job_list = data["retValue"]["recruitJobAdList"]
        total = data["retValue"]["total"]
        print(f"✅ 调用成功！共找到 {total} 个岗位，展示第1页：\n")

        for i, job in enumerate(job_list, 1):
            print(f"【第{i}个岗位】")
            print(f"岗位名称：{job.get('jobTitle')}")
            print(f"工作城市：{job.get('cityName')}")
            print(f"发布日期：{job.get('publishDate')}")
            print(f"所属部门：{job.get('buName')}")
            print("-" * 70)

    except Exception as e:
        print(f"❌ 程序异常：{str(e)}")

if __name__ == "__main__":
    get_ctrip_jobs()

✅ 调用成功！共找到 748 个岗位，展示第1页：

【第1个岗位】
岗位名称：AI产品经理(MJ034227)
工作城市：Shanghai
发布日期：2026-04-02
所属部门：Corporate Travel
----------------------------------------------------------------------
【第2个岗位】
岗位名称：Team Leader_EN_HTL_IB(MJ034219)
工作城市：Shanghai
发布日期：2026-04-02
所属部门：International business
----------------------------------------------------------------------
【第3个岗位】
岗位名称：高级/资深算法工程师（预测方向）(MJ021758)
工作城市：Shanghai
发布日期：2026-04-02
所属部门：Accommodation
----------------------------------------------------------------------
【第4个岗位】
岗位名称：后端开发工程师（大数据）(MJ026895)
工作城市：Shanghai
发布日期：2026-04-02
所属部门：Accommodation
----------------------------------------------------------------------
【第5个岗位】
岗位名称：高级/资深数据分析师（酒店业务）(MJ030243)
工作城市：Shanghai
发布日期：2026-04-02
所属部门：Accommodation
----------------------------------------------------------------------
【第6个岗位】
岗位名称：高级数据分析师（海外业务-base日本）(MJ033308)
工作城市：Shanghai
发布日期：2026-04-02
所属部门：Accommodation
----------------------------------------------------------------------
【第7个岗位

## 阿里系调试

### 淘天

In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# ===================== Edge 配置（不变） ======================
edge_options = webdriver.EdgeOptions()
edge_options.add_argument("--incognito")
edge_options.add_argument("--disable-blink-features=AutomationControlled")
edge_options.add_argument("--start-maximized")
edge_options.add_argument("--disable-popup-blocking")
edge_options.add_experimental_option("excludeSwitches", ["enable-automation"])
edge_options.add_experimental_option('useAutomationExtension', False)

# 初始化
driver = webdriver.Edge(options=edge_options)
wait = WebDriverWait(driver, 30)
driver.set_page_load_timeout(60)
TARGET_URL = "https://talent.taotian.com/off-campus/position-list?lang=zh"
job_data = []

# ===================== 自动筛选（已验证100%成功） ======================
def auto_filter():
    driver.get(TARGET_URL)
    print("页面加载中...")
    time.sleep(6)

    # 展开分类
    print("正在展开 运营类...")
    operate_expand = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, 'div[aria-label="运营类"] span.next-tree-switcher')))
    driver.execute_script("arguments[0].click();", operate_expand)
    time.sleep(1)

    print("正在展开 游戏类...")
    game_expand = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, 'div[aria-label="游戏类"] span.next-tree-switcher')))
    driver.execute_script("arguments[0].click();", game_expand)
    time.sleep(1)

    # 勾选岗位类型
    print("勾选 内容运营...")
    content_label = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@aria-label="内容运营"]/parent::span/parent::label')))
    driver.execute_script("arguments[0].click();", content_label)
    time.sleep(1)

    print("勾选 游戏运营...")
    game_label = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@aria-label="游戏运营"]/parent::span/parent::label')))
    driver.execute_script("arguments[0].click();", game_label)
    time.sleep(1)

    # 勾选城市
    cities = ["杭州", "上海", "深圳"]
    for city in cities:
        print(f"勾选 {city}...")
        city_label = wait.until(EC.element_to_be_clickable((By.XPATH, f'//input[@aria-label="{city}"]/parent::span/parent::label')))
        driver.execute_script("arguments[0].click();", city_label)
        time.sleep(1)
    
    print("✅ 筛选完成！\n")
    time.sleep(5)

# ===================== 提取岗位（严格按你的代码：URL + 职位要求） ======================
def extract_jobs():
    print("开始提取岗位信息...")
    # 你提供的正确class
    wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "_2AOmjKmlEtuR_KEoehWYcN")))
    job_items = driver.find_elements(By.CLASS_NAME, "_2AOmjKmlEtuR_KEoehWYcN")

    for idx, item in enumerate(job_items, 1):
        try:
            # 1. 列表页基础信息（你的代码）
            job_name = item.find_element(By.CLASS_NAME, "_1B0ew5wUb2dF_dhw0BFWtW").find_element(By.CLASS_NAME, "_3vj2eS7k7Mwpko5_6OSRu2").text.strip()
            inner_info_parent = item.find_element(By.CLASS_NAME, "_2qoSPTANtUY2-c4vLhjKv6")
            update_time = inner_info_parent.find_element(By.CLASS_NAME, "_3Jn5Z6PZA5H7Auzy0xlXu2").text.strip()

            # 2. 打开新标签页（你的代码）
            list_tab = driver.current_window_handle
            driver.execute_script("arguments[0].click();", item)
            time.sleep(3)
            new_tab = driver.window_handles[-1]
            driver.switch_to.window(new_tab)
            
            # 3. 获取详情URL + 职位要求（你的代码）
            detail_url = driver.current_url
            requirement = "无"
            content_blocks = driver.find_elements(By.CLASS_NAME, "content-block")
            if len(content_blocks) >= 3:
                requirement = content_blocks[2].text.strip().replace("\n", "；")

            # 4. 关闭标签，切回列表
            driver.close()
            driver.switch_to.window(list_tab)

            # 保存数据
            job_data.append({
                "岗位名称": job_name,
                "更新时间": update_time,
                "详情URL": detail_url,
                "职位要求": requirement
            })

        except Exception:
            # 异常跳过，不影响整体
            if len(driver.window_handles) > 1:
                driver.close()
                driver.switch_to.window(driver.window_handles[0])
            continue

    # ===================== 只打印前3条结果 ======================
    print("="*60)
    print("🎯 筛选结果（前3条）")
    print("="*60)
    for i, job in enumerate(job_data[:3], 1):
        print(f"\n【第{i}个岗位】")
        print(f"岗位名称：{job['岗位名称']}")
        print(f"更新时间：{job['更新时间']}")
        print(f"详情链接：{job['详情URL']}")
        print(f"职位要求：{job['职位要求'][:100]}...")  # 过长截断，方便查看
        print("-"*50)

# ===================== 主程序 ======================
if __name__ == "__main__":
    try:
        auto_filter()
        extract_jobs()
    finally:
        driver.quit()
        print("\n✅ 执行完毕！")

The msedgedriver version (145.0.3800.82) detected in PATH at .\msedgedriver.exe might not be compatible with the detected MicrosoftEdge version (146.0.3856.84); currently, msedgedriver 146.0.3856.84 is recommended for MicrosoftEdge 146.*, so it is advised to delete the driver in PATH and retry


页面加载中...
正在展开 运营类...
正在展开 游戏类...
勾选 内容运营...
勾选 游戏运营...
勾选 杭州...
勾选 上海...
勾选 深圳...
✅ 筛选完成！

开始提取岗位信息...
🎯 筛选结果（前3条）

【第1个岗位】
岗位名称：M&T事业部-美酒美食生鲜-店播运营
更新时间：更新于 2026-04-01
详情链接：https://talent.taotian.com/off-campus/position-detail?lang=zh&positionId=100011640022&track_id=SSP1775024566632QebYVtkmCP7404
职位要求：职位要求；1、有直播运营、电商运营相关经验；；2、具备商家运营能力，能运营商家的直播业务，对商家直播业务/流程/投资熟悉。；3、沟通能力强，能跨团队多业务的沟通；具备业务和资源整合能力，能够整合内外资...
--------------------------------------------------

【第2个岗位】
岗位名称：天猫事业部-直播间负责人-杭州
更新时间：更新于 2026-03-31
详情链接：https://talent.taotian.com/off-campus/position-detail?lang=zh&positionId=100012120022&track_id=SSP1775024566632dNlgtEICDH4550
职位要求：职位要求；1.本科及以上学历，具备5年以上头部电商平台直播间统筹操盘经验，有丰富的品牌合作经验；；2.具备直播电商全链路运营能力，熟悉从招商到运营到转化的各环节，数字感强，能够独立制定并执行经营策略；...
--------------------------------------------------

【第3个岗位】
岗位名称：淘天直播电商-中小达人运营-杭州
更新时间：更新于 2026-03-30
详情链接：https://talent.taotian.com/off-campus/position-detail?lang=zh&positionId=100010220023&track_id=SSP1775024566632NAZbOUViTh4513
职位要求：职位要求；1. 抖音、小红

## 米哈游调试

In [66]:
import requests
import time

# ===================== 固定配置 =====================
# 岗位列表接口
LIST_URL = "https://ats.openout.mihoyo.com/ats-portal/v1/job/list"
# 岗位详情接口（调试通过）
DETAIL_URL = "https://ats.openout.mihoyo.com/ats-portal/v1/job/info"

# 请求头
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": "https://jobs.mihoyo.com/",
    "Origin": "https://jobs.mihoyo.com",
    "Content-Type": "application/json;charset=UTF-8"
}

# 列表接口参数
LIST_PARAMS = {
    "jobName": "",
    "competencyTypes": [5, 6, 8],
    "channelDetailIds": [1, 2],
    "hireType": 0,
    "pageNo": 1,
    "pageSize": 50
}
# ====================================================

# 1. 获取岗位列表
def get_job_list():
    resp = requests.post(LIST_URL, headers=HEADERS, json=LIST_PARAMS)
    return resp.json()["data"]["list"]

# 2. 获取岗位详情（✅ 用调试通过的正确参数）
def get_job_detail(job_id):
    payload = {
        "id": job_id,
        "channelDetailIds": [1, 2]  # 必填！验证通过
    }
    resp = requests.post(DETAIL_URL, headers=HEADERS, json=payload)
    return resp.json()["data"]

# 3. 打印前3个岗位完整信息
if __name__ == "__main__":
    job_list = get_job_list()
    print(f"✅ 成功获取 {len(job_list)} 个岗位\n")

    # 打印前3个
    for i, job in enumerate(job_list[:3], 1):
        job_id = job["id"]
        detail = get_job_detail(job_id)
        
        print("="*80)
        print(f"【第{i}个岗位】")
        print(f"岗位名称：{detail['title']}")
        print(f"工作地点：{detail['addressDetailList'][0]['addressDetail']}")
        print(f"岗位类别：{detail['competencyType']}")
        print(f"详情链接：https://jobs.mihoyo.com/#/position/{job_id}")
        print("-"*50)
        print(f"📌 岗位职责：\n{detail['description']}")
        print("-"*30)
        print(f"📌 任职要求：\n{detail['jobRequire']}")
        
        time.sleep(0.3)

    print("\n🎉 前3个岗位详情打印完成！")

✅ 成功获取 50 个岗位

【第1个岗位】
岗位名称：AI 应用产品运营专家
工作地点：上海
岗位类别：运营类
详情链接：https://jobs.mihoyo.com/#/position/8289
--------------------------------------------------
📌 岗位职责：
1、AI效能项目全周期主导：负责公司级AI效能工具（如智能助手、自动化流程、知识管理AI化）的落地规划。独立完成从需求调研、价值评估、实施路径设计到效果复盘的全流程，确保项目与公司人效目标对齐；
2、跨部门复杂落地推动：主动深入各业务部门，诊断工作流痛点，设计AI解决方案的嵌入路径。解决在推广过程中遇到的流程冲突、习惯阻力与数据壁垒，驱动各部门按照既定节奏应用新工具，确保“真正用起来”；
3、运营机制与度量体系搭建：建立AI工具使用效果的追踪度量体系（如采纳率、活跃度、任务提效时长、满意度），通过数据洞察识别推广瓶颈与改进机会。设计运营策略（培训、案例、激励）来持续提升工具渗透率与使用深度；
4、内部赋能与知识沉淀：成为公司内部的“AI效能顾问”，为关键用户与团队提供深度赋能。沉淀不同场景的最佳实践、解决方案与排错指南，形成可复用的知识库，驱动效能的规模化提升。
------------------------------
📌 任职要求：
1、本科及以上学历，5-8年以上经验，有企业内部工具落地、效能运营、PMO或业务流程优化的直接经验，完整主导过企业级效率工具（如飞书、钉钉、自研平台）或自动化项目的推广，并对可量化的效能提升结果负责；
2、懂业务痛点：能快速理解研发、设计等不同部门的工作流与效能瓶颈，能精准判断AI在何处能产生最大价值；
3、强项目驱动：擅长在复杂组织中协调资源、管理干系人预期、突破推进阻力，有成熟的方法论确保项目闭环；
4、数据敏感：习惯用数据定义问题、评估效果，能设计简单的度量看板并从中发现洞察；
5、技术理解力：对LLM、Agent等AI技术如何具体解决办公与协同场景问题有较深认知，能与技术团队高效沟通。
【第2个岗位】
岗位名称：市场助理（海外社媒运营方向）-星布谷地（第三方编制）
工作地点：上海
岗位类别：国际化类
详情链接：https://jobs.mihoyo.com/#/position/8530
---------

## 字节调试

In [47]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
import time
import json
from datetime import datetime, timedelta

# ====================== 可自定义配置 ======================
# 目标招聘页URL
TARGET_URL = "https://jobs.bytedance.com/experienced/position?keywords=&category=&location=CT_125%2CCT_128%2CCT_52&project=&type=&job_hot_flag=&current=1&limit=10&functionCategory=&tag="
# 时间筛选：只保留近N天发布的岗位（0=不筛选，全量抓取）
DAYS_THRESHOLD = 7
# 可选：岗位关键词筛选（只保留标题含这些词的岗位，空列表=不筛选）
KEYWORD_FILTER = ["运营", "产品", "AI", "算法"]
# ===========================================================

# 浏览器防检测配置
edge_options = webdriver.EdgeOptions()
edge_options.add_argument("--incognito")
edge_options.add_argument("--disable-blink-features=AutomationControlled")
edge_options.add_argument("--start-maximized")
edge_options.add_experimental_option("excludeSwitches", ["enable-automation"])
edge_options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Edge(options=edge_options)
driver.implicitly_wait(10)

# ==============================================
# 接口拦截脚本（完全适配字节岗位API，Debug友好）
# ==============================================
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": """
        window.CAPTURE_API_DATA = null;
        window.LAST_API_URL = null;
        // 拦截Fetch
        const rawFetch = window.fetch;
        window.fetch = async (...args) => {
            const resp = await rawFetch(...args);
            if(args[0].includes('/api/v1/search/job/posts')){
                window.LAST_API_URL = args[0];
                const clone = resp.clone();
                window.CAPTURE_API_DATA = await clone.json();
                // 控制台打印日志，方便排查
                console.log('✅ 捕获到岗位接口数据', window.CAPTURE_API_DATA);
            }
            return resp;
        };
        // 拦截XHR兜底
        const rawXHR = window.XMLHttpRequest;
        window.XMLHttpRequest = function(){
            const xhr = new rawXHR();
            xhr.addEventListener('load', ()=>{
                if(xhr.responseURL.includes('/api/v1/search/job/posts')){
                    window.LAST_API_URL = xhr.responseURL;
                    try{
                        window.CAPTURE_API_DATA = JSON.parse(xhr.responseText);
                        console.log('✅ 捕获到岗位接口数据', window.CAPTURE_API_DATA);
                    }catch(e){
                        console.error('❌ 接口数据解析失败', e);
                    }
                }
            });
            return xhr;
        };
    """
})

# ==============================================
# 工具函数：时间判断+关键词筛选
# ==============================================
def is_job_valid(job):
    # 1. 时间筛选
    if DAYS_THRESHOLD > 0:
        try:
            publish_time_ms = job.get("publish_time", 0)
            publish_time = datetime.fromtimestamp(publish_time_ms / 1000)
            threshold_time = datetime.now() - timedelta(days=DAYS_THRESHOLD)
            if publish_time < threshold_time:
                return False, "发布时间超出范围"
        except Exception as e:
            print(f"⚠️ 时间解析失败，跳过校验：{e}")
    
    # 2. 关键词筛选
    if KEYWORD_FILTER:
        job_title = job.get("title", "").lower()
        has_keyword = any(keyword.lower() in job_title for keyword in KEYWORD_FILTER)
        if not has_keyword:
            return False, "标题不含目标关键词"
    
    return True, "符合条件"

# ==============================================
# 主抓取逻辑
# ==============================================
# 启动访问
driver.get(TARGET_URL)
time.sleep(5)  # 首次加载预留足够时间，确保接口返回

all_valid_jobs = []
page_num = 1
stop_crawl = False

while not stop_crawl:
    print(f"\n==================== 正在抓取第 {page_num} 页 ====================")
    
    # 1. 读取接口原始数据
    api_raw_data = driver.execute_script("return window.CAPTURE_API_DATA")
    
    # 【Debug兜底】每次翻页都保存原始数据到本地，确保不会丢数据
    with open("debug_raw_data.json", "w", encoding="utf-8") as f:
        json.dump(api_raw_data, f, indent=2, ensure_ascii=False)
    print(f"💾 本页原始数据已保存到 debug_raw_data.json")

    # 2. 解析岗位列表（完全对齐你提供的JSON结构）
    if not api_raw_data or api_raw_data.get("code") != 0:
        print("❌ 接口请求失败或无数据，终止抓取")
        break

    # 正确获取 data 节点
    data = api_raw_data.get("data", {})
    if "job_post_list" not in data:
        print("❌ 未获取到有效岗位列表，终止抓取")
        break

    job_list = data["job_post_list"] 
    total_count = data.get("total", 0)  
    print(f"📊 本页共 {len(job_list)} 个岗位，全站总计 {total_count} 个岗位")

    # 3. 逐个校验岗位
    page_valid_count = 0
    for job in job_list:
        job_title = job.get("title", "未知岗位")
        job_code = job.get("code", "无ID")
        job_city = job.get("city_info", {}).get("name", "未知城市")

        is_valid, reason = is_job_valid(job)
        if is_valid:
            all_valid_jobs.append(job)
            page_valid_count += 1
            print(f"✅ 保留 | {job_title} | {job_city} | 岗位ID：{job_code}")
        else:
            # 只有时间超范围才终止全量抓取（因为岗位是按发布时间倒序排的）
            if reason == "发布时间超出范围" and DAYS_THRESHOLD > 0:
                print(f"\n❌ 终止抓取 | 岗位【{job_title}】{reason}，后续岗位均为更早发布")
                stop_crawl = True
                break
            else:
                print(f"⚪ 跳过 | {job_title} | 原因：{reason}")

    print(f"📌 第 {page_num} 页处理完成，本页有效岗位：{page_valid_count} 个")

    # 4. 翻页逻辑（完全适配你提供的下一页按钮结构）
    if not stop_crawl:
        try:
            next_button = driver.find_element(
                By.XPATH,
                "//li[@title='下一页' and contains(@class, 'atsx-pagination-next') and @aria-disabled='false']"
            )
            # 点击前先重置数据，避免上一页数据干扰
            driver.execute_script("window.CAPTURE_API_DATA = null;")
            next_button.click()
            page_num += 1
            # 点击后预留足够时间，确保接口请求+返回完成
            time.sleep(4)
        except NoSuchElementException:
            print("\n🏁 已到达最后一页，无更多岗位，抓取完成！")
            break

# ==============================================
# 结果输出与保存
# ==============================================
print(f"\n🎉 全部抓取完成！总计筛选到有效岗位 {len(all_valid_jobs)} 个")
print("="*100)

# 1. 控制台打印精简汇总
for idx, job in enumerate(all_valid_jobs, 1):
    print(f"{idx}. {job.get('title')} | {job.get('city_info', {}).get('name')} | 岗位ID：{job.get('code')}")

# 2. 完整有效岗位数据保存
with open("bytedance_valid_jobs_full.json", "w", encoding="utf-8") as f:
    json.dump(all_valid_jobs, f, indent=2, ensure_ascii=False)
print(f"\n💾 完整有效岗位数据已保存到 bytedance_valid_jobs_full.json")

# 3. 可选：精简版数据保存（只保留核心字段，方便查看）
simplified_jobs = []
for job in all_valid_jobs:
    simplified_jobs.append({
        "岗位名称": job.get("title"),
        "岗位ID": job.get("code"),
        "工作城市": job.get("city_info", {}).get("name"),
        "岗位类型": job.get("job_category", {}).get("name"),
        "发布时间": datetime.fromtimestamp(job.get("publish_time", 0)/1000).strftime("%Y-%m-%d %H:%M:%S"),
        "岗位职责": job.get("description"),
        "任职要求": job.get("requirement"),
        "工作地址": job.get("job_post_info", {}).get("address")
    })

with open("bytedance_jobs_simplified.json", "w", encoding="utf-8") as f:
    json.dump(simplified_jobs, f, indent=2, ensure_ascii=False)
print("💾 精简版岗位数据已保存到 bytedance_jobs_simplified.json")

driver.quit()

The msedgedriver version (145.0.3800.82) detected in PATH at .\msedgedriver.exe might not be compatible with the detected MicrosoftEdge version (146.0.3856.84); currently, msedgedriver 146.0.3856.84 is recommended for MicrosoftEdge 146.*, so it is advised to delete the driver in PATH and retry



==================== 正在抓取第 1 页 ====================
💾 本页原始数据已保存到 debug_raw_data.json
📊 本页共 10 个岗位，全站总计 0 个岗位
✅ 保留 | 支付风控算法工程师-国际支付 | 深圳 | 岗位ID：A189598C

❌ 终止抓取 | 岗位【大模型平台产品经理-TikTok安全产品】发布时间超出范围，后续岗位均为更早发布
📌 第 1 页处理完成，本页有效岗位：1 个

🎉 全部抓取完成！总计筛选到有效岗位 1 个
1. 支付风控算法工程师-国际支付 | 深圳 | 岗位ID：A189598C

💾 完整有效岗位数据已保存到 bytedance_valid_jobs_full.json
💾 精简版岗位数据已保存到 bytedance_jobs_simplified.json


In [48]:
import datetime, pytz
def ts_to_date(ts):
    dt = datetime.datetime.fromtimestamp(ts/1000, pytz.UTC)
    return dt.astimezone(pytz.timezone('Asia/Shanghai')).strftime("%Y-%m-%d")
# 使用：ts_to_date(1774800000000) → "2026-03-30"

print(ts_to_date(1773221647972))

2026-03-11


## 小红书调试

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
import time
import json
from datetime import datetime, timedelta

TARGET_URL = "https://job.xiaohongshu.com/social/position?positionName=&jobTypes=om&workplaces=3100%2C4403%2C3301"

edge_options = webdriver.EdgeOptions()
edge_options.add_argument("--incognito")
edge_options.add_argument("--disable-blink-features=AutomationControlled")
edge_options.add_argument("--start-maximized")
edge_options.add_experimental_option("excludeSwitches", ["enable-automation"])

driver = webdriver.Edge(options=edge_options)
driver.implicitly_wait(10)

# ==============================================
# 原有CDP拦截脚本（无修改）
# ==============================================
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": """
        window.CAPTURE_API_DATA = null;
        // 拦截 Fetch
        const rawFetch = window.fetch;
        window.fetch = async (...args) => {
            const resp = await rawFetch(...args);
            if(args[0].includes('pageQueryPosition') || args[0].includes('social/position')){
                window.CAPTURE_API_DATA = await resp.clone().json();
            }
            return resp;
        };
        // 拦截 XMLHttpRequest
        const rawXHR = window.XMLHttpRequest;
        window.XMLHttpRequest = function(){
            const xhr = new rawXHR();
            xhr.addEventListener('load', ()=>{
                if(xhr.responseURL.includes('pageQueryPosition') || xhr.responseURL.includes('social/position')){
                    try{
                        window.CAPTURE_API_DATA = JSON.parse(xhr.responseText);
                    }catch{}
                }
            });
            return xhr;
        };
    """
})

# ==============================================
# 工具函数：判断时间是否为【近两天】
# ==============================================
def is_recent_two_days(updated_time_str):
    try:
        # 解析岗位更新时间（适配小红书时间格式：YYYY-MM-DD HH:MM:SS）
        update_time = datetime.strptime(updated_time_str, "%Y-%m-%d %H:%M:%S")
        # 当前时间 - 48小时 = 两天前的时间
        two_days_ago = datetime.now() - timedelta(days=2)
        # 大于两天前 = 近两天内
        return update_time >= two_days_ago
    except:
        # 时间解析失败，默认保留
        return True

# 访问第一页
driver.get(TARGET_URL)
time.sleep(3)

all_job_data = []
page_num = 1
# 终止标志：一旦触发，直接停止所有抓取
stop_crawl = False

while not stop_crawl:
    print(f"\n📄 正在抓取第 {page_num} 页数据...")
    
    api_data = driver.execute_script("return window.CAPTURE_API_DATA")
    
    if api_data and api_data.get("data") and api_data["data"].get("list"):
        job_list = api_data["data"]["list"]
        
        # 逐个遍历岗位（按时间倒序），筛选近两天
        for job in job_list:
            update_time_str = job.get("updateTime", "")
            is_recent = is_recent_two_days(update_time_str)
            
            if is_recent:
                all_job_data.append(job)
                print(f"✅ 保留岗位：{job.get('positionName')} | 更新时间：{update_time_str}")
            else:
                # 关键：遇到非近两天岗位，直接终止所有抓取！
                print(f"\n❌ 岗位【{job.get('positionName')}】更新时间不满足，已到最早时间，终止抓取！")
                stop_crawl = True
                break
        
        print(f"✅ 第 {page_num} 页处理完成，本页有效岗位：{len([j for j in job_list if is_recent_two_days(j.get('updateTime',''))])}")
    else:
        print(f"❌ 第 {page_num} 页未捕获到数据")
        break

    # ==============================================
    # 核心：按照你提供的HTML 【精准定位下一页按钮】
    # ==============================================
    if not stop_crawl:
        try:
            # 匹配：title=下一页 + 未禁用(aria-disabled=false)
            next_button = driver.find_element(
                By.XPATH, 
                "//li[@title='下一页' and @aria-disabled='false']"
            )
            next_button.click()
            page_num += 1
            # 重置拦截变量，避免上一页数据干扰
            driver.execute_script("window.CAPTURE_API_DATA = null;")
            time.sleep(3)
            
        except NoSuchElementException:
            print("\n🏁 已到达最后一页，抓取结束！")
            break

# ==============================================
# 输出结果 + 保存文件
# ==============================================
print(f"\n🎉 全部抓取完成！总计筛选到【近两天】岗位 {len(all_job_data)} 个")
print("="*60)
print(json.dumps(all_job_data, indent=2, ensure_ascii=False))

# 保存到本地
with open("xiaohongshu_recent_jobs.json", "w", encoding="utf-8") as f:
    json.dump(all_job_data, f, indent=2, ensure_ascii=False)
print("\n💾 近两天岗位数据已保存到 xiaohongshu_recent_jobs.json")

driver.quit()

## 鹰角调试

In [6]:
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import re
import os
from datetime import datetime, timedelta

# 驱动路径（同目录下的msedgedriver.exe）
DRIVER_PATH = "msedgedriver.exe"

def parse_post_time(time_text):
    """
    【简化版】仅适配鹰角固定时间格式：发布于 2025-12-10
    """
    # 直接提取固定格式中的日期
    date_match = re.search(r'发布于 (\d{4}-\d{1,2}-\d{1,2})', time_text)
    if date_match:
        return datetime.strptime(date_match.group(1), '%Y-%m-%d')
    # 无法解析则返回极早时间，直接过滤
    return datetime(2000, 1, 1)

def parse_job_description_from_html(html):
    """解析HTML字符串，兼容多关键词变体提取职位描述/要求"""
    # 初始化变量，修复未定义报错
    desc_key = ""
    req_key = ""

    # 移除所有HTML标签，保留文本和换行
    html = re.sub(r'<p.*?>', '\n', html)
    html = re.sub(r'</p>', '', html)
    html = re.sub(r'<ol.*?>', '\n', html)
    html = re.sub(r'</ol>', '', html)
    html = re.sub(r'<li.*?>', '\n', html)
    html = re.sub(r'</li>', '', html)
    html = re.sub(r'<br.*?>', '\n', html)
    html = re.sub(r'<.*?>', '', html)
    
    # 清理空白字符，去除空行
    lines = [line.strip() for line in html.split('\n') if line.strip()]
    pure_text = '\n'.join(lines)
    
    # 定义所有可能的描述类/要求类关键词
    desc_keywords = [
        "职位描述","工作职责","岗位职责","岗位描述","工作内容"
    ]
    req_keywords = [
        "任职要求","岗位要求","任职资格","职责要求","工作要求","职位要求"
    ]
    
    # 1. 找到第一个出现的描述类关键词
    desc_start = -1
    for key in desc_keywords:
        pos = pure_text.find(key)
        if pos != -1:
            desc_start = pos + len(key)
            desc_key = key
            break
    
    # 2. 找到第一个出现的要求类关键词
    req_start = -1
    if desc_start != -1:
        text_after_desc = pure_text[desc_start:]
        for key in req_keywords:
            pos = text_after_desc.find(key)
            if pos != -1:
                req_start = desc_start + pos + len(key)
                req_key = key
                break
    
    # 3. 分割描述和要求
    job_desc = ""
    job_req = ""
    if desc_start != -1 and req_start != -1:
        job_desc = pure_text[desc_start : req_start - len(req_key)].strip()
        job_req = pure_text[req_start:].strip()
    elif desc_start != -1:
        job_desc = pure_text[desc_start:].strip()
    elif req_start != -1:
        job_req = pure_text[req_start:].strip()
    
    # 清理空行
    job_desc = '\n'.join([line for line in job_desc.split('\n') if line.strip()])
    job_req = '\n'.join([line for line in job_req.split('\n') if line.strip()])
    
    return {
        "job_desc": job_desc,
        "job_req": job_req
    }

def extract_all_jobs_to_print():
    # 初始化浏览器
    edge_options = Options()
    edge_options.add_experimental_option("excludeSwitches", ["enable-logging"])
    
    if not os.path.exists(DRIVER_PATH):
        print(f"错误：未找到驱动文件！路径：{os.path.abspath(DRIVER_PATH)}")
        return
    
    driver = webdriver.Edge(service=Service(DRIVER_PATH), options=edge_options)
    
    # 筛选阈值：近两天
    now = datetime.now()
    two_days_ago = (now - timedelta(days=2)).replace(hour=0, minute=0, second=0, microsecond=0)
    print(f"⏰ 筛选条件：仅提取【{two_days_ago.strftime('%Y-%m-%d')}】之后发布的岗位\n")
    
    try:
        # ========== 1. 替换为你指定的官方URL ==========
        url = "https://jobs.hypergryph.com/apply/hypergryph/26325/#/jobs?page=1&commitment%5B0%5D=%E5%85%A8%E8%81%8C&zhineng%5B0%5D=46432&pageSize=15"
        driver.get(url)
        driver.maximize_window()
        wait = WebDriverWait(driver, 20)
        
        # 等待岗位列表加载
        wait.until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, ".container-aOp138AX_X.normal-TBuWTpDMcE.list-oR2doUijv4")
            )
        )
        time.sleep(3)
        
        # 提取岗位容器
        job_containers = driver.find_elements(
            By.CSS_SELECTOR, ".container-aOp138AX_X.normal-TBuWTpDMcE.list-oR2doUijv4"
        )
        total_jobs = len(job_containers)
        print(f"✅ 页面共找到 {total_jobs} 个岗位，开始筛选...\n")
        
        success_count = 0
        for idx, container in enumerate(job_containers, 1):
            try:
                # 滚动到岗位位置
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", container)
                time.sleep(0.5)

                # ========== 2. 固定时间元素选择器 + 固定格式解析 ==========
                job_time_elem = container.find_element(By.CSS_SELECTOR, ".published-at-PQ5IBWmbJV")
                job_time_text = job_time_elem.text.strip()
                job_update_time = parse_post_time(job_time_text)
                print(job_update_time)
                print(two_days_ago)
                
                # 超过两天直接终止（页面按时间倒序）
                if job_update_time < two_days_ago:
                    print(f"\n⏹️ 第 {idx} 个岗位已超过两天，停止提取！")
                    break
                
                # 提取岗位名称
                job_title_elem = container.find_element(
                    By.CSS_SELECTOR, ".title-u2qk9xX9Ie.target-color-container"
                )
                job_title = job_title_elem.text.strip()
                
                # 提取并解析岗位描述
                desc_elem = container.find_element(By.CLASS_NAME, "job-description-WwRmovZt9o")
                inner_html = desc_elem.get_attribute("innerHTML")
                parsed_info = parse_job_description_from_html(inner_html)
                job_desc = parsed_info["job_desc"]
                job_req = parsed_info["job_req"]
                
                success_count += 1
                
                # 格式化打印结果
                print("=" * 80)
                print(f"📌 序号：{success_count}")
                print(f"🏷️ 岗位名称：{job_title}")
                print(f"🕒 发布时间：{job_time_text}")
                print(f"\n📋 职位描述/职责：\n{job_desc if job_desc else '无'}")
                print(f"\n🔍 任职要求/资格：\n{job_req if job_req else '无'}")
                print("=" * 80 + "\n")
                
            except Exception as e:
                print(f"❌ 提取第 {idx} 个岗位失败：{str(e)[:100]}...\n")
                continue
        
        print(f"\n🎉 提取完成！共筛选出【{success_count}】个近两天的岗位")
        
    except Exception as e:
        print(f"\n❌ 程序执行出错：{str(e)}")
    finally:
        driver.quit()

if __name__ == "__main__":
    extract_all_jobs_to_print()

⏰ 筛选条件：仅提取【2026-03-30】之后发布的岗位

✅ 页面共找到 15 个岗位，开始筛选...

2026-04-01 00:00:00
2026-03-30 00:00:00
📌 序号：1
🏷️ 岗位名称：海外游戏内容运营（生态子公司）
🕒 发布时间：发布于 2026-04-01

📋 职位描述/职责：
1、负责运营团队紧急公告的视觉化处理，能够基于现有模板或快速建立新模板，在极短时间内完成公告图的修改、定稿与输出；
2、针对Discord等海外社群渠道，独立完成简单的攻略图解、UGC活动宣传图、表情包等轻量级内容的快速创意与制作；
3、根据游戏版本亮点、活动内容或内宣需求，细化执行脚本，完成游戏实机画面的录制，并运用剪辑软件进行后期剪辑、合成与包装，产出高质量的视频素材；
4、建立并维护可复用的内容素材库，持续优化工作流程，以提升紧急需求和常规内容的产出效率。

🔍 任职要求/资格：
1、熟练掌握图像处理软件（如 Photoshop）及主流视频剪辑软件（如 Premiere Pro、After Effects），具备独立、快速完成图文及视频内容制作的能力；
2、有责任心和抗压能力，具备良好的沟通能力，具备较好的审美能力；
3、热爱游戏行业，熟悉二次元游戏品类，了解海外用户的内容偏好者优先；
4、新媒体、广告设计、影视制作等相关专业优先；有游戏内容创作经验优先。
※面试前请提供个人作品集（包括视频、图文等案例，可附链接或提供脱敏文件）

2026-03-31 00:00:00
2026-03-30 00:00:00
📌 序号：2
🏷️ 岗位名称：海外社区用户运营（森空岛）
🕒 发布时间：发布于 2026-03-31

📋 职位描述/职责：
1、负责社区用户与内容运营工作，根据社区策略与游戏版本节奏，制定对应社区运营策略，包括但不限于策略分析、事件创意、排期制定、文案撰写、素材制作、舆情管理等；
2、负责海外官方社区用户体系搭建和内容长线运营；
3、负责游戏版本内容在社区上的活动宣发排期、传播落地及效果回收；
4、负责玩家舆情收集及汇报，关注品类竞品及新品动态；
5、负责对社区本地化工作进行需求下发和验收。

🔍 任职要求/资格：
1、本科及以上学历，3-5年游戏行业从业经验。热爱二次元，熟悉玩家群体；
2、熟悉海外社区平台的机制和功能，包含Facebo

## B站调试

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, ElementNotInteractableException, TimeoutException
import time
import datetime
from datetime import timedelta

# ===================== 基础配置 ======================
# 驱动路径（替换为你的驱动路径，比如msedgedriver.exe的绝对路径）
DRIVER_PATH = "msedgedriver.exe"
# B站招聘目标URL（热招岗位+code=03，可根据需要修改）
TARGET_URL = "https://jobs.bilibili.com/social/positions?code=03&type=3&page=1"

# 浏览器配置（禁用自动化检测，伪装真实浏览器）
edge_options = webdriver.EdgeOptions()
edge_options.add_argument("--disable-blink-features=AutomationControlled")
edge_options.add_argument("--start-maximized")  # 最大化窗口
edge_options.add_argument("--disable-popup-blocking")
# 伪装UA
edge_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36 Edg/123.0.0.0")
# 禁用selenium自动化特征
edge_options.add_experimental_option("excludeSwitches", ["enable-automation"])
edge_options.add_experimental_option("useAutomationExtension", False)

# 初始化驱动和显式等待
driver = webdriver.Edge(options=edge_options)
driver.set_page_load_timeout(60)
wait = WebDriverWait(driver, 10)

# 存储【近两天】的岗位数据
valid_job_data = []

# ===================== 时间筛选工具函数 ======================
def parse_release_time(time_str):
    """解析B站岗位发布时间为datetime对象（兼容所有B站时间格式）"""
    now = datetime.datetime.now()
    time_str = str(time_str).strip()

    # 格式1：2026-03-30
    if '-' in time_str:
        try:
            return datetime.datetime.strptime(time_str, '%Y-%m-%d')
        except ValueError:
            return now
    # 格式2：今天
    elif '今天' in time_str:
        return now
    # 格式3：昨天
    elif '昨天' in time_str:
        return now - timedelta(days=1)
    # 格式4：3天前 / 1天前
    elif '天前' in time_str:
        days = int(''.join(filter(str.isdigit, time_str)))
        return now - timedelta(days=days)
    # 异常格式
    else:
        return now

def is_recent_two_days(parsed_time):
    """判断时间是否在【近48小时/两天内】"""
    two_days_ago = datetime.datetime.now() - timedelta(days=8)
    return parsed_time >= two_days_ago

# ===================== 核心爬取函数 ======================
def crawl_job_detail(job_card, page_num, job_idx):
    """爬取单个岗位的完整信息"""
    job_info = {
        "页码": page_num,
        "页内序号": job_idx,
        "岗位名称": "未获取",
        "工作地点": "未获取",
        "岗位类别": "未获取",
        "工作性质": "未获取",
        "发布时间": "未获取",
        "工作职责": "无",
        "任职要求": "无",
        "详情页URL": "无"
    }

    try:
        # 1. 提取列表页基础信息
        try:
            job_name_elem = job_card.find_element(By.CLASS_NAME, "item-title")
            job_info["岗位名称"] = job_name_elem.text.strip()
        except NoSuchElementException:
            print(f"⚠️ 第{page_num}页第{job_idx}个岗位：未找到岗位名称")

        # 提取地点/类别/性质/发布时间
        try:
            infotags_elem = job_card.find_element(By.CLASS_NAME, "bili-infotags")
            span_list = infotags_elem.find_elements(By.TAG_NAME, "span")
            if len(span_list) >= 1:
                job_info["工作地点"] = span_list[0].text.strip()
            if len(span_list) >= 2:
                job_info["岗位类别"] = span_list[1].text.strip()
            if len(span_list) >= 3:
                job_info["工作性质"] = span_list[2].text.strip()
            if len(span_list) >= 4:
                job_info["发布时间"] = span_list[3].text.strip().replace("发布", "").strip()
        except NoSuchElementException:
            print(f"⚠️ 第{page_num}页第{job_idx}个岗位：未找到基础信息")

        # 2. 进入详情页
        try:
            list_tab = driver.current_window_handle
            driver.execute_script("arguments[0].click();", job_card)
            time.sleep(2)

            # 切换标签页
            if len(driver.window_handles) > 1:
                detail_tab = driver.window_handles[-1]
                driver.switch_to.window(detail_tab)
                job_info["详情页URL"] = driver.current_url

                # 提取职责要求
                try:
                    sub_title_elem = wait.until(
                        EC.presence_of_element_located((By.XPATH, "//p[@class='position-sub-title' and text()='职位描述']"))
                    )
                    desc_elem = sub_title_elem.find_element(By.XPATH, "./following-sibling::p[@class='position-desc'][1]")
                    desc_text = desc_elem.text.strip()

                    # 拆分内容
                    if "工作职责:" in desc_text:
                        resp_part = desc_text.split("工作职责:")[1]
                        job_info["工作职责"] = resp_part.split("工作要求:")[0].strip().replace("\n", "；") if "工作要求:" in resp_part else resp_part.strip().replace("\n", "；")
                    if "工作要求:" in desc_text:
                        job_info["任职要求"] = desc_text.split("工作要求:")[1].strip().replace("\n", "；")

                except NoSuchElementException:
                    print(f"⚠️ 第{page_num}页第{job_idx}个岗位：未找到职责要求")

                driver.close()
                driver.switch_to.window(list_tab)
            else:
                driver.back()
                time.sleep(2)
        except Exception as e:
            print(f"⚠️ 进入详情页失败：{str(e)}")

        print(f"✅ 第{page_num}页第{job_idx}个岗位爬取完成：{job_info['岗位名称']}")
        return job_info

    except Exception as e:
        print(f"❌ 岗位爬取异常：{str(e)}")
        return job_info

def crawl_single_page(page_num):
    """
    爬取单页岗位 + 时间筛选
    返回值：True=当前页全符合，继续下一页；False=遇到旧岗位，终止全部爬取
    """
    try:
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "space")))
        job_cards = driver.find_elements(By.CLASS_NAME, "bili-item-card")
        print(f"\n📄 第{page_num}页共找到 {len(job_cards)} 个岗位")

        if not job_cards:
            return False

        # 逐岗位处理（时间倒序，遇到旧岗位直接终止）
        for idx, card in enumerate(job_cards, 1):
            job_info = crawl_job_detail(card, page_num, idx)
            release_time = parse_release_time(job_info["发布时间"])

            # ✅ 核心筛选：近两天则保留，否则直接终止所有爬取
            if is_recent_two_days(release_time):
                valid_job_data.append(job_info)
            else:
                print(f"\n🚫 第{page_num}页第{idx}个岗位发布时间超出两天，终止爬取！")
                return False

            time.sleep(1)

        # 当前页所有岗位都符合条件
        return True

    except Exception as e:
        print(f"❌ 页面爬取失败：{str(e)}")
        return False

def click_next_page():
    """点击下一页"""
    try:
        next_btn = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, "ant-pagination-next")))
        if "disabled" in next_btn.get_attribute("class") or not next_btn.is_enabled():
            return False
        driver.execute_script("arguments[0].click();", next_btn)
        time.sleep(3)
        return True
    except:
        return False

# ===================== 打印结果函数 ======================
def print_job_results():
    """格式化打印所有近两天的岗位信息"""
    if not valid_job_data:
        print("\n❌ 未找到近两天发布的岗位信息")
        return

    total = len(valid_job_data)
    print(f"\n" + "="*80)
    print(f"🎉 爬取完成！共获取【近两天】岗位信息 {total} 条")
    print("="*80)

    for i, job in enumerate(valid_job_data, 1):
        print(f"\n🔹 第{i}条岗位信息")
        print(f"岗位名称：{job['岗位名称']}")
        print(f"工作地点：{job['工作地点']}")
        print(f"岗位类别：{job['岗位类别']}")
        print(f"工作性质：{job['工作性质']}")
        print(f"发布时间：{job['发布时间']}")
        print(f"详情链接：{job['详情页URL']}")
        print(f"工作职责：{job['工作职责']}")
        print(f"任职要求：{job['任职要求']}")
        print("-"*80)

# ===================== 主程序 ======================
if __name__ == "__main__":
    try:
        print("🚀 开始爬取B站【近两天】招聘岗位信息...")
        driver.get(TARGET_URL)
        time.sleep(3)

        current_page = 1
        while True:
            print(f"\n========== 处理第 {current_page} 页 ==========")
            # 爬取当前页，返回False则终止
            if not crawl_single_page(current_page):
                break
            # 无下一页则终止
            if not click_next_page():
                print("\n📌 已无下一页，爬取结束")
                break
            current_page += 1

        # 打印最终结果
        print_job_results()

    except Exception as e:
        print(f"\n💥 程序异常：{str(e)}")
    finally:
        driver.quit()
        print("\n🔌 浏览器已关闭")

## 网易调试

In [ ]:
import requests
import json
from typing import List, Dict
# 新增：时间处理模块，用于筛选近两天数据
from datetime import datetime, timedelta

def get_163_jobs() -> List[Dict]:
    """
    爬取网易招聘岗位信息，筛选近两天更新的岗位，直接打印结果
    :return: 包含近两天岗位信息的列表
    """
    job_list = []
    # 完整请求头
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Referer": "https://hr.163.com/job-list.html",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "zh-CN,zh;q=0.9",
        "Content-Type": "application/json;charset=UTF-8",
        "X-Requested-With": "XMLHttpRequest",
        "Origin": "https://hr.163.com"
    }
    
    api_url = "https://hr.163.com/api/hr163/position/queryPage"
    # 仅请求第1页，pageSize=200覆盖全部数据
    post_data = {
        "currentPage": 1,
        "pageSize": 200,
        "postType": "08",
        "workType": "0",
        "cityIdList":[229, 2, 138],
        "lang": "zh"
    }

    try:
        print("正在获取网易招聘岗位数据...")
        # 发送单次POST请求
        response = requests.post(
            api_url,
            headers=headers,
            json=post_data,
            timeout=15
        )
        response.raise_for_status()
        response_json = response.json()

        # 检查请求是否成功
        if response_json.get("code") != 200:
            print(f"请求失败: {response_json.get('msg')}")
            return job_list

        data = response_json.get("data")
        if not data:
            print("无数据返回")
            return job_list

        jobs = data.get("list", [])
        if not jobs:
            print("未获取到岗位信息")
            return job_list

        # 解析所有岗位信息（新增：更新时间戳，用于筛选）
        for job in jobs:
            job_id = job.get("id", "")
            detail_url = f"https://hr.163.com/job-detail.html?id={job_id}&lang=zh" if job_id else ""
            # 新增：获取岗位更新时间戳（毫秒级）
            update_time_stamp = job.get("updateTime", 0)
            
            job_info = {
                "岗位名称": job.get("name", ""),
                "岗位地址": ",".join(job.get("workPlaceNameList", [])),
                "职位描述": job.get("description", "").replace("\n", " "),
                "职位要求": job.get("requirement", "").replace("\n", " "),
                "岗位详情页URL": detail_url,
                "更新时间戳": update_time_stamp  # 用于时间筛选
            }
            job_list.append(job_info)

        # ===================== 核心修改：筛选近两天更新的岗位 =====================
        now = datetime.now()
        two_days_ago = now - timedelta(days=2)  # 计算48小时前的时间
        filtered_jobs = []
        
        for job in job_list:
            stamp = job["更新时间戳"]
            if not stamp:
                continue
            # 毫秒级时间戳转换为标准时间
            update_time = datetime.fromtimestamp(stamp / 1000)
            # 筛选：更新时间 >= 两天前
            if update_time >= two_days_ago:
                # 格式化时间，方便查看
                job["更新时间"] = update_time.strftime("%Y-%m-%d %H:%M:%S")
                filtered_jobs.append(job)
        # ======================================================================

        print(f"\n数据筛选完成！总数据：{len(job_list)} 条，近两天更新：{len(filtered_jobs)} 条\n")
        return filtered_jobs

    except requests.exceptions.RequestException as e:
        print(f"请求异常: {e}")
    except Exception as e:
        print(f"解析异常: {e}")

    return []

# 新增：格式化打印岗位信息
def print_jobs(jobs: List[Dict]):
    if not jobs:
        print("❌ 暂无近两天更新的岗位信息！")
        return
    
    # 遍历打印每个岗位，分隔线区分，清晰易读
    for index, job in enumerate(jobs, 1):
        print("-" * 80)
        print(f"【岗位 {index}】")
        print(f"岗位名称：{job['岗位名称']}")
        print(f"岗位地址：{job['岗位地址']}")
        print(f"更新时间：{job['更新时间']}")
        print(f"职位描述：{job['职位描述']}")
        print(f"职位要求：{job['职位要求']}")
        print(f"详情链接：{job['岗位详情页URL']}")
    print("-" * 80)

if __name__ == "__main__":
    # 1. 获取并筛选近两天的岗位数据
    filtered_job_data = get_163_jobs()
    # 2. 直接打印结果
    print_jobs(filtered_job_data)

## 大疆调试

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from datetime import date, timedelta
import time

def get_filtered_dji_jobs():
    # ===================== 核心配置 =====================
    # 大疆仅爬第一页
    BASE_URL = "https://we.dji.com/zh-CN/social?from=home_page&category=301_302&location=3100_4403&pageSize=100&page=1"
    # 排除关键词：硕士+工作年限
    EXCLUDE_KEYWORDS = ['硕士','3年','4年','5年','6年','7年','8年','9年','10年','三年','四年','五年']
    
    # ===================== 【你的代码风格】ARM Chromium 配置 =====================
    options = Options()
    # 必选参数（Docker+ARM 必备）
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")

    # 【关键】连接本地 Docker 中的 seleniarm 浏览器（容器间通信地址）
    driver = webdriver.Remote(
        command_executor="http://192.168.2.53:4444/wd/hub",
        options=options
    )
    driver.set_page_load_timeout(60)
    all_jobs = []

    # 获取近2天日期
    def get_valid_dates():
        today = date.today()
        yesterday = today - timedelta(days=1)
        return [today.strftime("%Y-%m-%d"), yesterday.strftime("%Y-%m-%d")]

    # 爬取岗位详情
    def crawl_job_detail(job_item):
        try:
            job_name = job_item.find_element(By.CLASS_NAME, "PositionCard_text__2BdZa").text.strip()
            keyword_text = job_item.find_element(By.CLASS_NAME, "PositionCard_keyword__FFaH5").text.strip()
            keyword_parts = [part.strip() for part in keyword_text.split("|")]
            city = keyword_parts[0]
            update_time = keyword_parts[-1]  # 直接获取日期：2026-03-18
            detail_url = job_item.find_element(By.TAG_NAME, "a").get_attribute("href")

            # 打开详情页
            main_handle = driver.current_window_handle
            driver.execute_script("window.open(arguments[0]);", detail_url)
            time.sleep(2)
            driver.switch_to.window(driver.window_handles[-1])
            time.sleep(2)

            # 提取任职要求
            requirement = ""
            subtitles = driver.find_elements(By.CLASS_NAME, "detail_subtitle__gOlwP")
            contents = driver.find_elements(By.CLASS_NAME, "detail_phases__PyEga")
            for i, sub in enumerate(subtitles):
                if "任职要求" in sub.text and i < len(contents):
                    requirement = contents[i].text.strip()

            driver.close()
            driver.switch_to.window(main_handle)
            return {
                "岗位名": job_name,
                "工作地点": city,
                "详情链接": detail_url,
                "更新时间": update_time,
                "岗位要求": requirement
            }
        except Exception:
            if len(driver.window_handles) > 1:
                driver.close()
                driver.switch_to.window(driver.window_handles[0])
            return None

    # ===================== 主爬取逻辑 =====================
    try:
        valid_dates = get_valid_dates()
        driver.get(BASE_URL)
        time.sleep(5)
        
        # 获取岗位列表
        job_items = driver.find_elements(By.CLASS_NAME, "social_position_card__epffd")
        
        for item in job_items:
            job = crawl_job_detail(item)
            if job:
                all_jobs.append(job)

        # 双重筛选
        recent_jobs = [j for j in all_jobs if j["更新时间"] in valid_dates]
        final_jobs = [j for j in recent_jobs if not any(k in j["岗位要求"] for k in EXCLUDE_KEYWORDS)]
        
        return final_jobs
    finally:
        driver.quit()

# 运行并输出结果
if __name__ == "__main__":
    result = get_filtered_dji_jobs()
    print(f"\n✅ 筛选完成，符合条件岗位：{len(result)}")
    for job in result:
        print(job)